# Silver Layer — Dimensional Modelling

**Objective:** Transform the bronze flat file into a star-schema data model.
Four dimension tables (calendar, supplier, region, product) and one fact table
(weekly sales) are created, cleaned, and saved as Parquet.

**Input :** `data/bronze/iqvia-sales_data.parquet`  
**Output:** `data/silver/dim_*.parquet` + `data/silver/fact_sales_weekly.parquet`

In [ ]:
import pandas as pd

In [ ]:
# 1. Load bronze
df = pd.read_parquet('../../data/bronze/iqvia-sales_data.parquet')
print(f"{len(df):,} rows  |  date range: {df['week_date'].min().date()} to {df['week_date'].max().date()}")

## dim_calendar

Daily calendar spanning the full date range, enriched with temporal attributes
(year, semester, quarter, month, week). Week dates are anchored to Monday
for unambiguous weekly joins.


In [ ]:
# Build daily spine and derive all temporal attributes
start_date = df['week_date'].dt.to_period('W').apply(lambda r: r.start_time).min()

df_dim_time = pd.DataFrame({'date': pd.date_range(start=start_date, end=df['week_date'].max(), freq='D')})
df_dim_time['year']             = df_dim_time['date'].dt.year
df_dim_time['semester']         = df_dim_time['date'].dt.month.apply(lambda x: 1 if x <= 6 else 2)
df_dim_time['semester_date']    = pd.to_datetime(df_dim_time['date'].dt.to_period('6M').apply(lambda r: r.start_time))
df_dim_time['semester_name']    = df_dim_time['semester'].apply(lambda x: 'H1' if x == 1 else 'H2')
df_dim_time['quarter']          = df_dim_time['date'].dt.quarter
df_dim_time['quarter_date']     = pd.to_datetime(df_dim_time['date'].dt.to_period('Q').apply(lambda r: r.start_time))
df_dim_time['quarter_name']     = df_dim_time['quarter'].apply(lambda x: f'Q{x}')
df_dim_time['month']            = df_dim_time['date'].dt.month
df_dim_time['month_name']       = df_dim_time['date'].dt.month_name()
df_dim_time['month_date']       = pd.to_datetime(df_dim_time['date'].dt.to_period('M').apply(lambda r: r.start_time))
df_dim_time['day']              = df_dim_time['date'].dt.day
df_dim_time['week']             = df_dim_time['date'].dt.isocalendar().week
df_dim_time['week_date']        = pd.to_datetime(df_dim_time['date'].dt.to_period('W').apply(lambda r: r.start_time))
df_dim_time['day_of_week']      = df_dim_time['date'].dt.dayofweek
df_dim_time['day_of_week_name'] = df_dim_time['date'].dt.day_name()

df_dim_time.to_parquet('../../data/silver/dim_calendar.parquet', index=False)
print(f"dim_calendar: {df_dim_time.shape[0]:,} days  — Saved")
df_dim_time.head()

## dim_supplier

One row per supplier. Supplier names are anonymised (e.g. Supplier A).


In [ ]:
df_dim_supplier = pd.DataFrame({'supplier_id': df['supplier_id'].sort_values().unique()})
df_dim_supplier['supplier_name'] = df_dim_supplier['supplier_id'].apply(lambda x: f'Supplier {x}')

df_dim_supplier.to_parquet('../../data/silver/dim_supplier.parquet', index=False)
print(f"dim_supplier: {len(df_dim_supplier)} suppliers  — Saved")


## dim_region

One row per Brazilian macro-region (IBGE standard). Region IDs are assigned alphabetically starting at 1.


In [ ]:
df_dim_region = pd.DataFrame(df['region'].sort_values().unique()).reset_index()
df_dim_region.columns = ['region_id', 'region_name']
df_dim_region['region_id'] += 1  # 1-indexed

df_dim_region.to_parquet('../../data/silver/dim_region.parquet', index=False)
print(f"dim_region: {len(df_dim_region)} regions  — Saved")
df_dim_region


## dim_product

One row per product. Three anonymised hierarchy levels are preserved:
`product_attribute_1` (family) → `product_attribute_2` (line) → `product_attribute_3` (class).


In [ ]:
df_dim_product = (
    df[['product_id', 'product_attribute_1', 'product_attribute_2', 'product_attribute_3']]
    .drop_duplicates()
    .sort_values('product_id')
    .reset_index(drop=True)
)
df_dim_product['product_name'] = df_dim_product['product_id'].apply(lambda x: f'Product {x}')

df_dim_product.to_parquet('../../data/silver/dim_product.parquet', index=False)
print(f"dim_product: {len(df_dim_product)} products  — Saved")


## fact_sales_weekly

Joins the bronze flat table with `dim_region` to replace raw region names with integer IDs and standardises the week date to Monday.
Multiple rows for the same (week, supplier, product, region) are summed — consistent with weekly sell-out data.


In [ ]:
# Join to get Monday-anchored week_date and integer region_id
df_fact_sales = (
    df[['week_date', 'supplier_id', 'product_id', 'region', 'units_sold']]
    .rename(columns={'week_date': 'week_date_raw'})
    .merge(df_dim_time[['date', 'week_date']], left_on='week_date_raw', right_on='date', how='left')
    .drop(columns=['date', 'week_date_raw'])
    .merge(df_dim_region[['region_id', 'region_name']], left_on='region', right_on='region_name', how='left')
    .drop(columns=['region', 'region_name'])
)

# Aggregate duplicate (week, supplier, product, region) rows by summing units
df_fact_sales = (
    df_fact_sales
    .groupby(['week_date', 'supplier_id', 'product_id', 'region_id'])
    .agg(units_sold=('units_sold', 'sum'))
    .reset_index()
)

# Enforce column order and add a surrogate key
df_fact_sales = (
    df_fact_sales[['week_date', 'supplier_id', 'region_id', 'product_id', 'units_sold']]
    .reset_index(drop=True)
    .assign(sales_id=lambda d: d.index + 1)
)

df_fact_sales.to_parquet('../../data/silver/fact_sales_weekly.parquet', index=False)
print(f"fact_sales_weekly: {len(df_fact_sales):,} rows  — Saved")
df_fact_sales.head()
